# T Cell Only Ref-Query Merge Pipeline v1.3
**Changelog v1.3 (debug-friendly ipynb):**
- `ensure_counts_layer`: BUG-6 完整修复 — sparse `[:1000]` 现在正确调用 `.toarray().ravel()`
- `label_order` 从 `predict(soft=True)` 的 DataFrame columns 提取，不再使用 `scanvi_model.labels_`（该属性返回逐细胞预测而非类别名）
- 去除 `main()` 包装，所有 Step 可逐 cell 独立运行
- 新增双 scANVI 分支：保留 reference-label scANVI，同时增加 `CellTypist + v1.1 的 CD4/CD8 score` 组合标签 scANVI，并将两套结果一并保存

## Cell 0 — Imports & Global Settings

In [23]:
import sys
import warnings
import json
import gc
import joblib
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
from scipy import sparse
from scipy.stats import entropy

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import torch
import scanpy as sc
import scvi
import celltypist
from celltypist import models
from umap import UMAP

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

print("=" * 80)
print("T Cell ONLY Ref-Query Merge Pipeline v1.3")
print("=" * 80)

gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

scvi.settings.dl_num_workers = 0
print(f"scVI dl_num_workers: {scvi.settings.dl_num_workers}")

T Cell ONLY Ref-Query Merge Pipeline v1.3
GPU available: True
GPU Device: Tesla V100-SXM2-16GB
scVI dl_num_workers: 0


## Cell 1 — Configuration
> **运行前必须修改 `REFERENCE_H5AD` 和 `QUERY_H5AD`**

In [24]:
# 1. CONFIGURATION
# ==============================================================================

REFERENCE_H5AD = "/home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad"
QUERY_H5AD = "/home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/t_cells.h5ad"

REF_LABEL_COARSE = "cell_type_L2"
REF_LABEL_FINE = "cell_type_L3"

BATCH_KEY = "sample"
TISSUE_KEY = "tissue"

OUTPUT_DIR = "/home/h2048/data/py/20260225/tcell_only_merged_pipeline"
OUTPUT_PREFIX = "tcell_only_merged"

INCLUDE_COARSE_TYPES = None

N_HVG = 4000
FORCE_MARKERS_IN_HVG = True

SCVI_N_LATENT = 100
SCVI_N_LAYERS = 2
SCVI_N_HIDDEN = 128
SCVI_DROPOUT = 0.1
MAX_EPOCHS_SCVI = 400

MAX_EPOCHS_SCANVI = 200
UNLABELED_CATEGORY = "Unknown"

BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0

CELLTYPIST_MODEL = "/home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl"
CELLTYPIST_MAJORITY_VOTE = True
CELLTYPIST_SCANVI_USE_MAJORITY = True
CELLTYPIST_SCANVI_LABEL_KEY = "scanvi_labels_celltypist_cd4cd8"
CELLTYPIST_SCANVI_RESULT_KEY = "celltypist_cd4cd8"
CELLTYPIST_SCANVI_KEEP_DP_DN_UNPREFIXED = True

QUERY_LEIDEN_RESOLUTION = 1.0

CD4_SCORE_THRESHOLD = 0.3
CD8_SCORE_THRESHOLD = 0.3

TCELL_CORE_MARKERS = ["CD3D", "CD3E", "CD3G", "PTPRC"]
CD4_MARKERS = ["CD4", "IL7R", "CD40LG"]
CD8_MARKERS = ["CD8A", "CD8B"]
NAIVE_MARKERS = ["CCR7", "SELL", "TCF7", "LEF1", "CD27"]
CM_MARKERS = ["CCR7", "CD27", "IL7R"]
EM_MARKERS = ["GZMK", "CXCR3", "CCR5"]
TEMRA_MARKERS = ["GZMB", "PRF1", "GNLY", "NKG7"]
TREG_MARKERS = ["FOXP3", "IL2RA", "CTLA4", "IKZF2"]
TH1_MARKERS = ["TBX21", "IFNG", "CXCR3"]
TH2_MARKERS = ["GATA3", "IL4", "IL5", "IL13"]
TH17_MARKERS = ["RORC", "IL17A", "IL17F", "CCR6"]
TRM_MARKERS = ["CD69", "ITGAE", "CXCR6"]
PROLIF_MARKERS = ["MKI67", "TOP2A", "PCNA"]

STRESS_SIGNATURE_GENES = [
    "HSPA1A", "HSPA1B", "HSPA8", "HSP90AA1", "HSP90AB1", "DNAJB1",
    "JUN", "JUNB", "JUND", "FOS", "FOSB", "EGR1", "IER2"
]

S_GENES = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UHRF1",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "HELLS", "RFC2", "RPA2",
    "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7",
    "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1"
]

G2M_GENES = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80",
    "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "FAM64A",
    "SMC4", "CCNB1", "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E"
]

FORCED_MARKERS = list(set(
    TCELL_CORE_MARKERS + CD4_MARKERS + CD8_MARKERS +
    NAIVE_MARKERS + CM_MARKERS + EM_MARKERS + TEMRA_MARKERS +
    TREG_MARKERS + TH1_MARKERS + TH2_MARKERS + TH17_MARKERS +
    TRM_MARKERS + PROLIF_MARKERS
))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED
scvi.settings.seed = RANDOM_SEED

Seed set to 42


## Cell 2 — Helper Functions (all `def`s)

In [25]:
# 2. HELPER FUNCTIONS
# ==============================================================================

def ensure_counts_layer(adata, counts_layer="counts"):
    """Robust counts validation with float32 tolerance."""
    if counts_layer not in (adata.layers or {}):
        print(f"  WARNING: layers['{counts_layer}'] not found, checking .X...")
        if hasattr(adata, 'X') and adata.X is not None:
            # BUG-6 FIX: sparse[:N] returns sub-sparse-matrix, not values.
            # Must call .toarray().ravel(). Dense path: np.asarray handles np.matrix.
            if issparse(adata.X):
                X_sample = adata.X[:1000].toarray().ravel()
            else:
                X_sample = np.asarray(adata.X[:1000]).ravel()
            sample = np.asarray(X_sample, dtype=np.float64)

            if np.any(sample < 0):
                raise ValueError("adata.X contains negative values - not valid raw counts!")

            if np.allclose(sample, np.round(sample), atol=1e-6):
                print(f"  -> Auto-copying .X to layers['{counts_layer}']")
                if issparse(adata.X) and not isinstance(adata.X, csr_matrix):
                    adata.layers[counts_layer] = csr_matrix(adata.X)
                else:
                    adata.layers[counts_layer] = adata.X.copy()
            else:
                raise ValueError(
                    f"CRITICAL ERROR: layers['{counts_layer}'] not found and .X "
                    f"does not look like raw counts (non-integer values). "
                    f"Sample range: [{sample.min():.4f}, {sample.max():.4f}]"
                )
        else:
            raise ValueError(
                f"CRITICAL ERROR: layers['{counts_layer}'] not found and .X is None. "
                "scVI requires raw counts!"
            )

    X_counts = adata.layers[counts_layer]
    if issparse(X_counts):
        sample_data = X_counts.data[:1000]
    else:
        sample_data = X_counts.flat[:1000]

    sample = np.asarray(sample_data, dtype=np.float64)

    if np.any(sample < 0):
        raise ValueError("counts contains negative values!")

    if not np.allclose(sample, np.round(sample), atol=1e-6):
        raise ValueError(
            "counts looks non-integer (possible normalized/log data). "
            f"Sample range: [{sample.min():.4f}, {sample.max():.4f}]"
        )

    if issparse(X_counts) and not isinstance(X_counts, csr_matrix):
        adata.layers[counts_layer] = csr_matrix(X_counts)

    return counts_layer


def ensure_batch_tissue(adata, batch_key, tissue_key):
    """Ensure batch and tissue columns exist and are categorical."""
    if batch_key not in adata.obs.columns:
        print(f"  WARNING: {batch_key} not found, creating placeholder")
        adata.obs[batch_key] = "unknown_batch"
    adata.obs[batch_key] = adata.obs[batch_key].astype("category")

    if tissue_key not in adata.obs.columns:
        print(f"  WARNING: {tissue_key} not found, creating placeholder")
        adata.obs[tissue_key] = "unknown_tissue"
    adata.obs[tissue_key] = adata.obs[tissue_key].astype("category")
    if "unknown_tissue" not in adata.obs[tissue_key].cat.categories:
        adata.obs[tissue_key] = adata.obs[tissue_key].cat.add_categories(["unknown_tissue"])
    adata.obs[tissue_key] = adata.obs[tissue_key].fillna("unknown_tissue")


def compute_module_score_efficient(adata, gene_list, score_name, counts_layer="counts"):
    """
    Memory-efficient module score computation.

    BUG-1 FIX (P0): The original code passed gene_list = adata_tmp.var_names
    (all genes in the subset) to sc.tl.score_genes. Since adata_tmp was built
    to contain ONLY the marker genes, there were no background control genes
    left, causing sc.tl.score_genes to raise a ValueError or return 0 for
    every cell. Replaced with direct normalized mean-expression score, which
    is the correct and well-defined approach when working with a small marker
    subset.
    """
    genes = [g for g in gene_list if g in adata.var_names]
    if len(genes) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    gene_idx = adata.var_names.get_indexer(genes)
    gene_idx = gene_idx[gene_idx >= 0]

    if len(gene_idx) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    # Extract only the marker gene subset
    X_subset = adata.layers[counts_layer][:, gene_idx]
    var_subset = adata.var.iloc[gene_idx].copy()

    adata_tmp = sc.AnnData(X=X_subset.copy(), var=var_subset)
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    # --- BUG-1 FIX ---
    # Old (broken): sc.tl.score_genes(adata_tmp, gene_list=gene_names, ...)
    # Explanation: adata_tmp only contains the target genes, so there are no
    # control background genes. sc.tl.score_genes randomly samples ctrl genes
    # from var_names; when gene_list == var_names, the ctrl set is empty and
    # the function raises ValueError or returns all-zero scores.
    #
    # New (correct): compute mean normalized expression across the marker set.
    # This is equivalent to the "score" concept without needing background genes.
    X_norm = adata_tmp.X
    if issparse(X_norm):
        mean_expr = np.asarray(X_norm.mean(axis=1)).flatten()
    else:
        mean_expr = np.asarray(X_norm).mean(axis=1).flatten()

    adata.obs[score_name] = mean_expr
    # --- END FIX ---

    # Normalize to 0-1 for thresholding
    scores = adata.obs[score_name]
    mn, mx = float(scores.min()), float(scores.max())
    if mx > mn:
        adata.obs[f"{score_name}_norm"] = (scores - mn) / (mx - mn)
    else:
        adata.obs[f"{score_name}_norm"] = 0.0

    del adata_tmp, X_subset
    gc.collect()


def compute_cd4_cd8_scores(adata):
    """Compute CD4 and CD8 module scores for validation."""
    print("  -> Computing CD4/CD8 module scores...")

    compute_module_score_efficient(adata, CD4_MARKERS, "CD4_score")
    compute_module_score_efficient(adata, CD8_MARKERS, "CD8_score")

    cd4_high = adata.obs["CD4_score_norm"] > CD4_SCORE_THRESHOLD
    cd8_high = adata.obs["CD8_score_norm"] > CD8_SCORE_THRESHOLD

    classifications = []
    for i in range(len(adata)):
        cd4 = cd4_high.iloc[i]
        cd8 = cd8_high.iloc[i]
        if cd4 and not cd8:
            classifications.append("CD4_single")
        elif cd8 and not cd4:
            classifications.append("CD8_single")
        elif cd4 and cd8:
            classifications.append("DP")
        else:
            classifications.append("DN")

    adata.obs["cd4_cd8_by_score"] = classifications
    print(f"    CD4_single: {sum([c == 'CD4_single' for c in classifications])}")
    print(f"    CD8_single: {sum([c == 'CD8_single' for c in classifications])}")
    print(f"    DP: {sum([c == 'DP' for c in classifications])}")
    print(f"    DN: {sum([c == 'DN' for c in classifications])}")


def prepare_covariates(adata):
    """Prepare covariates for scVI training."""
    print("  -> Validating counts...")
    ensure_counts_layer(adata, "counts")

    print("  -> Batch/Tissue...")
    ensure_batch_tissue(adata, BATCH_KEY, TISSUE_KEY)

    print("  -> Computing signature scores...")
    if "pct_counts_mt" not in adata.obs.columns:
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")

    compute_cd4_cd8_scores(adata)
    compute_module_score_efficient(adata, STRESS_SIGNATURE_GENES, "stress_score")

    if not all(k in adata.obs.columns for k in ["S_score", "G2M_score"]):
        s_in = [g for g in S_GENES if g in adata.var_names]
        g_in = [g for g in G2M_GENES if g in adata.var_names]

        if len(s_in) >= 5 and len(g_in) >= 5:
            cc_union = list(dict.fromkeys(s_in + g_in))
            cc_idx = adata.var_names.get_indexer(cc_union)
            X_cc = adata.layers["counts"][:, cc_idx].copy()
            var_cc = adata.var.iloc[cc_idx].copy()

            ad_tmp = sc.AnnData(X=X_cc, var=var_cc)
            sc.pp.normalize_total(ad_tmp, target_sum=1e4)
            sc.pp.log1p(ad_tmp)
            sc.tl.score_genes_cell_cycle(ad_tmp, s_genes=s_in, g2m_genes=g_in)

            adata.obs["S_score"] = ad_tmp.obs["S_score"].values
            adata.obs["G2M_score"] = ad_tmp.obs["G2M_score"].values
            adata.obs["phase"] = ad_tmp.obs["phase"].values

            del ad_tmp
            gc.collect()
        else:
            adata.obs["S_score"] = 0.0
            adata.obs["G2M_score"] = 0.0
            adata.obs["phase"] = "G1"


def run_celltypist_on_full_genes(adata_merged, hvg_mask, model_name=CELLTYPIST_MODEL, majority_vote=True):
    """
    Run CellTypist on full gene matrix (not HVG subset).

    BUG-2 FIX (P0): The original code called
        models.download_models(model=model_name)
    where model_name is a full file path like
        "/home/h2048/.../Immune_All_Low.pkl".
    download_models() expects a model name ("Immune_All_Low.pkl"), not a path.
    Passing a path causes an HTTP error (it tries to construct an invalid URL)
    or silently does nothing while Model.load() then also fails because the
    keyword argument semantics differ from path loading.

    Fix: if the given string is an existing file path, load directly with
    Model.load(model_name). Only call download_models (with basename) when
    the local file does not exist.
    """
    print("\n[CellTypist] Starting annotation on FULL gene matrix...")

    # --- BUG-2 FIX ---
    # Old (broken):
    #   models.download_models(model=model_name)  # fails with full path
    #   model = models.Model.load(model_name)
    #
    # New (correct): check if local file exists first.
    if os.path.isfile(model_name):
        print(f"  -> Loading model from local path: {model_name}")
        model = models.Model.load(model_name)
    else:
        model_basename = os.path.basename(model_name)
        print(f"  -> Local model not found, downloading: {model_basename}")
        models.download_models(model=model_basename)
        model = models.Model.load(model_basename)
    # --- END FIX ---

    model_genes = set(model.features)

    available_genes = [g for g in adata_merged.var_names if g in model_genes]
    print(f"  -> {len(available_genes)}/{len(model_genes)} model genes present in data")

    if len(available_genes) < 100:
        raise ValueError(f"Too few overlapping genes ({len(available_genes)}) for CellTypist!")

    adata_ct = adata_merged[:, available_genes].copy()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    predictions = celltypist.annotate(
        adata_ct,
        model=model,
        majority_voting=majority_vote,
        mode='best match'
    )

    pl = predictions.predicted_labels

    if isinstance(pl, pd.DataFrame):
        if "predicted_labels" in pl.columns:
            pred_labels = pl["predicted_labels"].astype(str).values
        else:
            pred_labels = pl.iloc[:, 0].astype(str).values

        conf_values = predictions.probability_matrix.max(axis=1).values

        majority_labels = None
        if majority_vote and "majority_voting" in pl.columns:
            majority_labels = pl["majority_voting"].astype(str).values
    else:
        pred_labels = pl.astype(str).values
        conf_values = predictions.probability_matrix.max(axis=1).values
        majority_labels = None

    adata_merged.obs["celltypist_pred"] = pred_labels
    adata_merged.obs["celltypist_confidence"] = conf_values

    if majority_labels is not None:
        adata_merged.obs["celltypist_majority"] = majority_labels

    print(f"  -> CellTypist predictions added to full matrix")
    print(pd.Series(pred_labels).value_counts().head(10))

    del adata_ct
    gc.collect()
    return predictions


def compute_novelty_scores(adata_merged, proba_df):
    """
    Compute novelty scores for query cells based on prediction entropy.
    High entropy = uncertain = potentially novel.
    """
    print("  -> Computing novelty scores (entropy-based)...")

    proba_array = proba_df.values
    epsilon = 1e-10
    entropies = entropy(proba_array + epsilon, axis=1)

    adata_merged.obs["scanvi_entropy"] = entropies

    ent_min, ent_max = entropies.min(), entropies.max()
    if ent_max > ent_min:
        adata_merged.obs["novelty_score"] = (entropies - ent_min) / (ent_max - ent_min)
    else:
        adata_merged.obs["novelty_score"] = 0.0

    query_mask = adata_merged.obs["data_source"] == "query"
    high_novelty = (adata_merged.obs["novelty_score"] > 0.7) & query_mask
    adata_merged.obs["is_potentially_novel"] = high_novelty

    print(f"    High novelty query cells: {high_novelty.sum()}")


def run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION):
    """
    Run Leiden clustering on query cells only using scVI latent space.

    BUG-4 FIX (P1): The original code indexed the numpy array
        adata_merged.obsm["X_scVI"][query_mask]
    where query_mask is a pandas boolean Series. Indexing a numpy array with
    a pandas Series uses the Series' integer index internally; when the
    Series index is not 0,1,2,... (which is the case after concat with
    string obs_names), numpy falls back to object-array indexing and may
    silently select wrong cells or raise an IndexError.
    Fix: use query_mask.values to pass a plain numpy boolean array.
    """
    print(f"\n[Novelty Detection] Running query-only Leiden (resolution={resolution})...")

    query_mask = adata_merged.obs["data_source"] == "query"
    query_cells = adata_merged.obs_names[query_mask]

    if len(query_cells) < 10:
        print("  -> Too few query cells, skipping query-only clustering")
        adata_merged.obs["leiden_query"] = "N/A"
        return

    # --- BUG-4 FIX ---
    # Old (broken): adata_merged.obsm["X_scVI"][query_mask]
    # query_mask is a pandas Series; numpy array indexing with a pandas Series
    # is position-unsafe when obs_names are non-integer strings.
    # New (correct): use .values to obtain a plain numpy boolean array.
    X_scVI_query = adata_merged.obsm["X_scVI"][query_mask.values]
    # --- END FIX ---

    adata_qry_tmp = sc.AnnData(
        X=X_scVI_query,
        obs=adata_merged.obs.loc[query_cells].copy()
    )
    adata_qry_tmp.obsm["X_scVI"] = X_scVI_query

    sc.pp.neighbors(adata_qry_tmp, use_rep="X_scVI", n_neighbors=30, random_state=RANDOM_SEED)
    sc.tl.leiden(adata_qry_tmp, resolution=resolution, random_state=RANDOM_SEED)

    leiden_full = pd.Series("N/A", index=adata_merged.obs_names, dtype="object")
    leiden_full.loc[query_cells] = "qry_" + adata_qry_tmp.obs["leiden"].astype(str)
    adata_merged.obs["leiden_query"] = leiden_full.values

    print(f"  -> Found {adata_qry_tmp.obs['leiden'].nunique()} query-only clusters")
    print(adata_merged.obs["leiden_query"].value_counts().head(10))

    del adata_qry_tmp
    gc.collect()


def ensure_label_category(adata, label_key, unlabeled_category=UNLABELED_CATEGORY):
    """Ensure a label column is categorical and contains the unlabeled category."""
    adata.obs[label_key] = adata.obs[label_key].astype(str).astype("category")
    if unlabeled_category not in adata.obs[label_key].cat.categories:
        adata.obs[label_key] = adata.obs[label_key].cat.add_categories([unlabeled_category])
    return label_key


def choose_celltypist_source_key(adata, prefer_majority=True):
    """Choose CellTypist majority voting labels when available, else predicted labels."""
    if prefer_majority and "celltypist_majority" in adata.obs.columns:
        return "celltypist_majority"
    if "celltypist_pred" in adata.obs.columns:
        return "celltypist_pred"
    raise KeyError("CellTypist labels not found in adata.obs")


def is_non_t_like_label(label):
    """Avoid forcing CD4/CD8 prefixes onto clearly non-T labels such as NK."""
    label_lower = str(label).lower()
    non_t_keywords = ["nk", "natural killer", "ilc"]
    return any(keyword in label_lower for keyword in non_t_keywords)


def build_celltypist_cd4cd8_scanvi_labels(
    adata,
    output_key=CELLTYPIST_SCANVI_LABEL_KEY,
    prefer_majority=CELLTYPIST_SCANVI_USE_MAJORITY,
    score_key="cd4_cd8_by_score",
):
    """Combine CellTypist labels with v1.1 CD4/CD8 score-based prefixes for an alternative scANVI branch."""
    if score_key not in adata.obs.columns:
        raise KeyError(f"Missing score key: {score_key}")

    source_key = choose_celltypist_source_key(adata, prefer_majority=prefer_majority)
    base_labels = adata.obs[source_key].astype(str)
    score_labels = adata.obs[score_key].astype(str)

    combined = []
    for base_label, score_label in zip(base_labels, score_labels):
        label_clean = str(base_label).strip()
        label_upper = label_clean.upper()

        if is_non_t_like_label(label_clean):
            combined.append(label_clean)
            continue

        if score_label == "CD4_single" and "CD4" not in label_upper:
            combined.append(f"CD4+ {label_clean}")
        elif score_label == "CD8_single" and "CD8" not in label_upper:
            combined.append(f"CD8+ {label_clean}")
        elif not CELLTYPIST_SCANVI_KEEP_DP_DN_UNPREFIXED and score_label in {"DP", "DN"}:
            combined.append(f"{score_label} {label_clean}")
        else:
            combined.append(label_clean)

    adata.obs["celltypist_scanvi_source_label"] = base_labels.values
    adata.obs["celltypist_scanvi_combined_label"] = pd.Series(
        combined,
        index=adata.obs_names,
        dtype="object"
    )
    adata.obs[output_key] = adata.obs["celltypist_scanvi_combined_label"].copy()
    ensure_label_category(adata, output_key, unlabeled_category=UNLABELED_CATEGORY)

    print(f"  -> CellTypist source labels for alternate scANVI: {source_key}")
    print(f"  -> Alternate label distribution ({output_key}):")
    print(adata.obs[output_key].value_counts().head(15))

    return output_key, source_key


def get_scanvi_soft_predictions(scanvi_model, adata_input):
    """Version-safe extraction of soft predictions and label order."""
    proba_raw = scanvi_model.predict(adata_input, soft=True)
    if isinstance(proba_raw, pd.DataFrame):
        label_order = list(proba_raw.columns)
        proba = proba_raw.values.astype(np.float32)
    else:
        proba = np.asarray(proba_raw, dtype=np.float32)
        try:
            label_order = list(
                scanvi_model.adata_manager
                .get_state_registry("labels")
                .categorical_mapping
            )
        except Exception:
            label_order = [f"label_{i}" for i in range(proba.shape[1])]
    return proba, label_order


def train_scanvi_branch(scvi_model, adata_train, labels_key, branch_name, train_kwargs):
    """Train one scANVI branch from a shared scVI backbone."""
    if labels_key not in adata_train.obs.columns:
        raise KeyError(f"Missing labels_key in adata_train.obs: {labels_key}")

    ensure_label_category(adata_train, labels_key, unlabeled_category=UNLABELED_CATEGORY)

    print(f"\n  -> Initializing scANVI branch: {branch_name}")
    print(f"     labels_key = {labels_key}")
    print(f"     unique labels = {adata_train.obs[labels_key].nunique()}")
    print(adata_train.obs[labels_key].value_counts().head(15))

    model = scvi.model.SCANVI.from_scvi_model(
        scvi_model,
        adata=adata_train,
        labels_key=labels_key,
        unlabeled_category=UNLABELED_CATEGORY
    )
    model.train(**train_kwargs)
    print(f"  -> scANVI complete: {branch_name}")
    return model


def export_scanvi_branch_results(
    scanvi_model,
    adata_train,
    adata_merged,
    result_suffix,
    labels_key,
    set_as_default=False,
    compute_novelty=False,
):
    """Export latent space, predictions, probabilities and metadata for one scANVI branch."""
    named_latent_key = f"X_scANVI_{result_suffix}"
    named_pred_key = f"scanvi_pred_{result_suffix}"
    named_conf_key = f"scanvi_confidence_{result_suffix}"
    named_proba_key = f"scanvi_proba_{result_suffix}"
    named_order_key = f"scanvi_label_order_{result_suffix}"

    latent_scanvi = scanvi_model.get_latent_representation(adata_train)
    latent_df = pd.DataFrame(
        latent_scanvi,
        index=adata_train.obs_names,
        columns=[f"scANVI_{result_suffix}_{i}" for i in range(latent_scanvi.shape[1])]
    )
    latent_aligned = latent_df.reindex(adata_merged.obs_names)

    if latent_aligned.isna().any().any():
        raise ValueError(f"CRITICAL: Missing latent representation for branch '{result_suffix}'")

    adata_merged.obsm[named_latent_key] = latent_aligned.values

    pred_labels = scanvi_model.predict(adata_train)
    pred_df = pd.Series(pred_labels, index=adata_train.obs_names)
    pred_aligned = pred_df.reindex(adata_merged.obs_names)
    adata_merged.obs[named_pred_key] = pred_aligned.values

    proba, label_order = get_scanvi_soft_predictions(scanvi_model, adata_train)
    proba_df = pd.DataFrame(proba, index=adata_train.obs_names, columns=label_order)
    proba_aligned = proba_df.reindex(adata_merged.obs_names)
    adata_merged.obsm[named_proba_key] = proba_aligned.values
    adata_merged.obs[named_conf_key] = proba_aligned.values.max(axis=1)
    adata_merged.uns[named_order_key] = list(label_order)

    adata_merged.uns[f"scanvi_branch_{result_suffix}"] = {
        "labels_key": labels_key,
        "latent_key": named_latent_key,
        "prediction_key": named_pred_key,
        "confidence_key": named_conf_key,
        "probability_key": named_proba_key,
        "label_order_key": named_order_key,
    }

    if set_as_default:
        adata_merged.obsm["X_scANVI"] = adata_merged.obsm[named_latent_key].copy()
        adata_merged.obs["scanvi_pred"] = adata_merged.obs[named_pred_key].values
        adata_merged.obs["scanvi_confidence"] = adata_merged.obs[named_conf_key].values
        adata_merged.obsm["scanvi_proba"] = adata_merged.obsm[named_proba_key].copy()
        adata_merged.uns["scanvi_label_order"] = list(label_order)

    if compute_novelty:
        compute_novelty_scores(adata_merged, proba_aligned)

    return {
        "latent_key": named_latent_key,
        "prediction_key": named_pred_key,
        "confidence_key": named_conf_key,
        "probability_key": named_proba_key,
        "label_order_key": named_order_key,
        "label_order": label_order,
    }

## Cell 3 — Initialization (before Step 1)

In [26]:
"""Main pipeline function."""

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

## Cell 4 — Step 1: Load Reference and Query

In [27]:
print("\n[Step 1] Loading Reference and Query...")

print(f"  Loading reference: {REFERENCE_H5AD}")
adata_ref = sc.read_h5ad(REFERENCE_H5AD)
adata_ref.var_names_make_unique()
print(f"    Reference shape: {adata_ref.shape}")

print(f"  Loading query: {QUERY_H5AD}")
adata_qry = sc.read_h5ad(QUERY_H5AD)
adata_qry.var_names_make_unique()
print(f"    Query shape: {adata_qry.shape}")


[Step 1] Loading Reference and Query...
  Loading reference: /home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad
    Reference shape: (44833, 34252)
  Loading query: /home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/t_cells.h5ad
    Query shape: (145335, 83690)


## Cell 5 — Step 2: Subset Reference if needed

In [28]:
if INCLUDE_COARSE_TYPES is not None and REF_LABEL_COARSE in adata_ref.obs.columns:
    print(f"\n[Step 2] Subsetting reference to: {INCLUDE_COARSE_TYPES}")
    mask = adata_ref.obs[REF_LABEL_COARSE].isin(INCLUDE_COARSE_TYPES)
    adata_ref = adata_ref[mask].copy()
    print(f"    Reference after subset: {adata_ref.shape}")

## Cell 6 — Step 3: Prepare labels

In [29]:
print("\n[Step 3] Preparing labels...")

if REF_LABEL_COARSE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_coarse"] = adata_ref.obs[REF_LABEL_COARSE].astype(str)
    print(f"  -> Coarse labels from: {REF_LABEL_COARSE}")
    print(adata_ref.obs["cell_type_coarse"].value_counts())
else:
    print(f"  WARNING: {REF_LABEL_COARSE} not found, using 'T_cell'")
    adata_ref.obs["cell_type_coarse"] = "T_cell"

if REF_LABEL_FINE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs[REF_LABEL_FINE].astype(str)
    print(f"  -> Fine labels from: {REF_LABEL_FINE}")
else:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs["cell_type_coarse"]
    print(f"  -> Using coarse labels as fine labels")

adata_qry.obs["cell_type_coarse"] = UNLABELED_CATEGORY
adata_qry.obs["cell_type_fine"] = UNLABELED_CATEGORY


[Step 3] Preparing labels...
  -> Coarse labels from: cell_type_L2
cell_type_coarse
CD16+ NK cells                      10963
Trm cytotoxic T cells                9500
CD8+ Trm cytotoxic T cells           5412
Tem/Effector helper T cells          3845
Tem/Trm cytotoxic T cells            3652
NK cells                             2847
CD8+ Tem/Trm cytotoxic T cells       2123
Tem/Temra cytotoxic T cells          1705
CD8+ Tem/Temra cytotoxic T cells     1642
Regulatory T cells                    967
Type 17 helper T cells                335
MAIT cells                            298
Follicular helper T cells             253
CD8+ Tem/Effector helper T cells      216
CD8+ gamma-delta T cells              194
ILC3                                  180
Tcm/Naive helper T cells              142
CD4+ Regulatory T cells               140
gamma-delta T cells                   131
Type 1 helper T cells                 104
CD16- NK cells                         94
CD4+ Tem/Effector helper T cells 

## Cell 7 — Step 4: Find common genes (deterministic order)

In [30]:
print("\n[Step 4] Finding common genes...")

qry_set = set(adata_qry.var_names)
common_genes = [g for g in adata_ref.var_names if g in qry_set]

print(f"  Reference: {adata_ref.n_vars:,}, Query: {adata_qry.n_vars:,}, Common: {len(common_genes):,}")

if len(common_genes) < 1000:
    raise ValueError(f"Too few common genes ({len(common_genes)}). Check gene naming!")

adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()


[Step 4] Finding common genes...
  Reference: 34,252, Query: 83,690, Common: 33,749


## Cell 8 — Step 5: Validate counts

In [31]:
print("\n[Step 5] Validating counts...")
ensure_counts_layer(adata_ref, "counts")
ensure_counts_layer(adata_qry, "counts")


[Step 5] Validating counts...
  -> Auto-copying .X to layers['counts']


'counts'

## Cell 9 — Step 6: Concatenate

In [32]:
print("\n[Step 6] Concatenating...")

adata_ref.obs_names = pd.Index([f"ref_{x}" for x in adata_ref.obs_names])
adata_qry.obs_names = pd.Index([f"qry_{x}" for x in adata_qry.obs_names])

adata_merged = sc.concat(
    {"reference": adata_ref, "query": adata_qry},
    axis=0,
    join="inner",
    merge="unique",
    label="data_source"
)

print(f"  Merged: {adata_merged.shape}")
print(f"  Reference: {(adata_merged.obs['data_source'] == 'reference').sum():,}")
print(f"  Query: {(adata_merged.obs['data_source'] == 'query').sum():,}")

del adata_ref, adata_qry
gc.collect()


[Step 6] Concatenating...
  Merged: (190168, 33749)
  Reference: 44,833
  Query: 145,335


14349

## Cell 10 — Step 7: Prepare covariates

In [33]:
print("\n[Step 7] Preparing covariates...")
prepare_covariates(adata_merged)

print("  -> Adding symbol_base column...")
adata_merged.var["symbol_base"] = adata_merged.var_names.str.replace(r"-\d+$", "", regex=True)


[Step 7] Preparing covariates...
  -> Validating counts...
  -> Batch/Tissue...
  -> Computing signature scores...
  -> Computing CD4/CD8 module scores...
    CD4_single: 96750
    CD8_single: 0
    DP: 0
    DN: 93418
  -> Adding symbol_base column...


## Cell 11 — Step 8: HVG Selection

In [34]:
print("\n[Step 8] Selecting HVGs...")
hvg_method = "unknown"
try:
    sc.pp.highly_variable_genes(
        adata_merged, layer="counts", n_top_genes=N_HVG,
        batch_key=BATCH_KEY, flavor="seurat_v3", subset=False
    )
    hvg_method = "batch_seurat_v3"
except Exception as e1:
    print(f"  -> batch-aware failed ({str(e1)[:50]}), trying standard...")
    try:
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="seurat_v3", subset=False
        )
        hvg_method = "standard_seurat_v3"
    except Exception as e2:
        print(f"  -> standard failed ({str(e2)[:50]}), fallback to cell_ranger")
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="cell_ranger", subset=False
        )
        hvg_method = "cell_ranger"

print(f"  -> Method: {hvg_method}")

if FORCE_MARKERS_IN_HVG:
    n_added = 0
    marker_set = set(FORCED_MARKERS)
    for idx, symbol_base in enumerate(adata_merged.var["symbol_base"]):
        if symbol_base in marker_set:
            real_name = adata_merged.var_names[idx]
            if not adata_merged.var.loc[real_name, "highly_variable"]:
                adata_merged.var.loc[real_name, "highly_variable"] = True
                n_added += 1
    print(f"  -> Forced {n_added}/{len(FORCED_MARKERS)} markers into HVG")

n_hvg_final = adata_merged.var["highly_variable"].sum()
print(f"  -> Final HVG count: {n_hvg_final}")

hvg_genes = adata_merged.var_names[adata_merged.var["highly_variable"]].tolist()
with open(output_dir / f"{OUTPUT_PREFIX}_hvg_genes.txt", "w") as f:
    f.write("\n".join(hvg_genes))


[Step 8] Selecting HVGs...
  -> batch-aware failed (b'There are other near singularities as well. 0.09), trying standard...
  -> Method: standard_seurat_v3
  -> Forced 13/41 markers into HVG
  -> Final HVG count: 4013


## Cell 12 — Step 9: Build FULL matrix for .raw

In [35]:
print("\n[Step 9] Building full matrix for .raw...")
full_counts = adata_merged.layers["counts"]
if issparse(full_counts) and not isinstance(full_counts, csr_matrix):
    full_counts = csr_matrix(full_counts)
raw_var = adata_merged.var.copy()
print(f"  Full matrix shape: {full_counts.shape}")


[Step 9] Building full matrix for .raw...
  Full matrix shape: (190168, 33749)


## Cell 13 — Step 10: Create training subset

In [36]:
print("\n[Step 10] Creating training subset...")

hvg_mask = adata_merged.var["highly_variable"].values
X_hvg = adata_merged.layers["counts"][:, hvg_mask]

if issparse(X_hvg) and not isinstance(X_hvg, csr_matrix):
    X_hvg = csr_matrix(X_hvg)

adata_train = sc.AnnData(
    X=X_hvg.copy(),
    obs=adata_merged.obs.copy(),
    var=adata_merged.var.iloc[hvg_mask].copy()
)
adata_train.var_names = adata_merged.var_names[hvg_mask]
adata_train.layers["counts"] = adata_train.X

print(f"  Training data: {adata_train.shape}")

adata_train.obs["scanvi_labels"] = adata_train.obs["cell_type_fine"].astype(str)
ensure_label_category(adata_train, "scanvi_labels")

# Placeholder for the alternate CellTypist + CD4/CD8 branch.
# Step 11 will overwrite this with real CellTypist-derived labels when available.
adata_train.obs[CELLTYPIST_SCANVI_LABEL_KEY] = adata_train.obs["scanvi_labels"].astype(str)
ensure_label_category(adata_train, CELLTYPIST_SCANVI_LABEL_KEY)

print("  -> Reference-label distribution:")
print(adata_train.obs["scanvi_labels"].value_counts())

gc.collect()


[Step 10] Creating training subset...
  Training data: (190168, 4013)
  -> Reference-label distribution:
scanvi_labels
Unknown                                   145335
cd16plus_nk_cells_c0                        5023
cd16plus_nk_cells_c1                        4215
trm_cytotoxic_t_cells_c0                    4058
trm_cytotoxic_t_cells_c1                    2993
cd8plus_trm_cytotoxic_t_cells_c0            2509
trm_cytotoxic_t_cells_c2                    2449
tem_effector_helper_t_cells_c0              2005
tem_trm_cytotoxic_t_cells_c0                1932
tem_effector_helper_t_cells_c1              1840
cd16plus_nk_cells_c2                        1725
tem_trm_cytotoxic_t_cells_c1                1720
nk_cells_c0                                 1590
cd8plus_trm_cytotoxic_t_cells_c1            1569
cd8plus_tem_trm_cytotoxic_t_cells_c0        1337
cd8plus_trm_cytotoxic_t_cells_c2            1334
nk_cells_c1                                 1257
tem_temra_cytotoxic_t_cells_c0               98

0

In [45]:
print("\n[Checkpoint] Logging AnnData structures before CellTypist...")
from datetime import datetime


def summarize_anndata_structure(adata, name):
    lines = [
        f"[{name}]",
        f"  shape: {adata.shape}",
        f"  X type: {type(adata.X).__name__}",
        f"  obs columns ({len(adata.obs.columns)}): {list(adata.obs.columns)}",
        f"  var columns ({len(adata.var.columns)}): {list(adata.var.columns)}",
        f"  layers: {list(adata.layers.keys())}",
        f"  obsm: {list(adata.obsm.keys())}",
        f"  varm: {list(adata.varm.keys())}",
        f"  obsp: {list(adata.obsp.keys())}",
        f"  uns: {list(adata.uns.keys())}",
        f"  raw present: {adata.raw is not None}",
    ]
    if adata.raw is not None:
        lines.append(f"  raw shape: {adata.raw.shape}")
    return lines


log_lines = [
    f"=== {datetime.now().isoformat(timespec='seconds')} :: AnnData structure checkpoint ===",
    f"output_dir: {output_dir}",
    f"output_h5ad: {output_h5ad if 'output_h5ad' in globals() else 'N/A'}",
    "",
]

for adata_name in ["adata_merged", "adata_train"]:
    if adata_name in globals():
        log_lines.extend(summarize_anndata_structure(globals()[adata_name], adata_name))
    else:
        log_lines.append(f"[{adata_name}] not found in globals()")
    log_lines.append("")

log_text = "\n".join(log_lines)
print(log_text)

log_path = output_dir / f"{OUTPUT_PREFIX}_anndata_structure.log"
with open(log_path, "a", encoding="utf-8") as f:
    f.write(log_text + "\n")

print(f"\n  -> AnnData structure log appended to: {log_path}")


[Checkpoint] Logging AnnData structures before CellTypist...
=== 2026-03-09T07:13:51 :: AnnData structure checkpoint ===
output_dir: /home/h2048/data/py/20260225/tcell_only_merged_pipeline
output_h5ad: /home/h2048/data/py/20260225/tcell_only_merged_pipeline/tcell_only_merged_results.h5ad

[adata_merged]
  shape: (190168, 33749)
  X type: csr_matrix
  obs columns (87): ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'age', 'sex', 'percent.mt', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', '

## Cell 14 — Step 11: CellTypist (full genes, backfill to training subset)

In [37]:
print("\n[Step 11] Running CellTypist on full gene matrix...")
try:
    run_celltypist_on_full_genes(adata_merged, hvg_mask)
    adata_train.obs["celltypist_pred"] = adata_merged.obs.loc[adata_train.obs_names, "celltypist_pred"]
    adata_train.obs["celltypist_confidence"] = adata_merged.obs.loc[adata_train.obs_names, "celltypist_confidence"]
    if "celltypist_majority" in adata_merged.obs.columns:
        adata_train.obs["celltypist_majority"] = adata_merged.obs.loc[adata_train.obs_names, "celltypist_majority"]

    alt_label_key, alt_source_key = build_celltypist_cd4cd8_scanvi_labels(
        adata_merged,
        output_key=CELLTYPIST_SCANVI_LABEL_KEY,
        prefer_majority=CELLTYPIST_SCANVI_USE_MAJORITY,
    )

    for col in ["celltypist_scanvi_source_label", "celltypist_scanvi_combined_label", alt_label_key]:
        adata_train.obs[col] = adata_merged.obs.loc[adata_train.obs_names, col].values
    ensure_label_category(adata_train, alt_label_key)

    print(f"  -> Alternate scANVI labels ready from: {alt_source_key}")
    print(adata_train.obs[alt_label_key].value_counts().head(15))
except Exception as e:
    print(f"  WARNING: CellTypist failed: {e}")
    import traceback
    traceback.print_exc()
    print("  -> Falling back to reference fine labels for alternate scANVI branch")
    adata_merged.obs[CELLTYPIST_SCANVI_LABEL_KEY] = adata_merged.obs["cell_type_fine"].astype(str)
    ensure_label_category(adata_merged, CELLTYPIST_SCANVI_LABEL_KEY)
    adata_train.obs[CELLTYPIST_SCANVI_LABEL_KEY] = adata_train.obs["scanvi_labels"].astype(str)
    ensure_label_category(adata_train, CELLTYPIST_SCANVI_LABEL_KEY)


[Step 11] Running CellTypist on full gene matrix...

[CellTypist] Starting annotation on FULL gene matrix...
  -> Loading model from local path: /home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl
  -> 6236/6639 model genes present in data


🔬 Input data has 190168 cells and 6236 genes
🔗 Matching reference genes in the model
🧬 6236 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


  -> CellTypist predictions added to full matrix
Tem/Trm cytotoxic T cells      34444
Trm cytotoxic T cells          25438
Regulatory T cells             24311
CD16+ NK cells                 17576
Tem/Effector helper T cells    15383
Tem/Temra cytotoxic T cells    13673
CD16- NK cells                  8480
Tcm/Naive helper T cells        8048
NK cells                        6901
Type 1 helper T cells           6890
Name: count, dtype: int64
  -> CellTypist source labels for alternate scANVI: celltypist_majority
  -> Alternate label distribution (scanvi_labels_celltypist_cd4cd8):
scanvi_labels_celltypist_cd4cd8
Trm cytotoxic T cells               21826
Tem/Trm cytotoxic T cells           20882
CD4+ Trm cytotoxic T cells          20185
CD4+ Regulatory T cells             18548
CD16+ NK cells                      17216
CD4+ Tem/Trm cytotoxic T cells      16470
CD4+ Tem/Effector helper T cells    15748
CD16- NK cells                       8730
Tem/Temra cytotoxic T cells          8140
Regu

## Cell 15 — Step 12: scVI Training

In [38]:
print("\n[Step 12] Training scVI...")

setup_kwargs = {
    "layer": "counts",
    "batch_key": BATCH_KEY,
    "continuous_covariate_keys": ["pct_counts_mt", "stress_score", "S_score", "G2M_score"],
    "categorical_covariate_keys": [TISSUE_KEY]
}

scvi.model.SCVI.setup_anndata(adata_train, **setup_kwargs)

scvi_model = scvi.model.SCVI(
    adata_train,
    n_latent=SCVI_N_LATENT,
    n_layers=SCVI_N_LAYERS,
    n_hidden=SCVI_N_HIDDEN,
    dropout_rate=SCVI_DROPOUT
)

train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCVI,
    "batch_size": BATCH_SIZE,
    "early_stopping": True,
    "early_stopping_patience": 30,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    train_kwargs["accelerator"] = "gpu"
    train_kwargs["devices"] = 1

scvi_model.train(**train_kwargs)
print("  -> scVI complete")


[Step 12] Training scVI...
SCVI model with the following parameters: 
n_hidden: 128, n_latent: 100, n_layers: 2, dropout_rate: 0.1, dispersion: gene, 
gene_likelihood: zinb, latent_distribution: normal.
Training status: Trained
Model's adata is minified?: False


ScanVI Model with the following params: 
unlabeled_category: Unknown, n_hidden: 128, n_latent: 100, n_layers: 2, 
dropout_rate: 0.1, dispersion: gene, gene_likelihood: zinb
Training status: Trained
Model's adata is minified?: False
SCVI model with the following parameters: 
n_hidden: 128, n_latent: 100, n_layers: 2, dropout_rate: 0.1, dispersion: gene, 
gene_likelihood: zinb, latent_distribution: normal.
Training status: Trained
Model's adata is minified?: False


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 391/400:  98%|█████████▊| 391/400 [2:25:11<03:20, 22.28s/it, v_num=1, train_loss_step=625, train_loss_epoch=659]  
Monitored metric elbo_validation did not improve in the last 30 records. Best score: 664.290. Signaling Trainer to stop.
  -> scVI complete


## Cell 16 — Step 13: scANVI Training

In [39]:
print("\n[Step 13] Training scANVI branches...")

scanvi_train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCANVI,
    "batch_size": BATCH_SIZE,
    "early_stopping": True,
    "early_stopping_patience": 20,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    scanvi_train_kwargs["accelerator"] = "gpu"
    scanvi_train_kwargs["devices"] = 1

scanvi_models = {}
scanvi_models["reference"] = train_scanvi_branch(
    scvi_model,
    adata_train,
    labels_key="scanvi_labels",
    branch_name="reference_fine_labels",
    train_kwargs=scanvi_train_kwargs,
 )
scanvi_model = scanvi_models["reference"]

scanvi_models[CELLTYPIST_SCANVI_RESULT_KEY] = train_scanvi_branch(
    scvi_model,
    adata_train,
    labels_key=CELLTYPIST_SCANVI_LABEL_KEY,
    branch_name=CELLTYPIST_SCANVI_RESULT_KEY,
    train_kwargs=scanvi_train_kwargs,
 )
scanvi_model_celltypist_cd4cd8 = scanvi_models[CELLTYPIST_SCANVI_RESULT_KEY]

print("  -> Dual scANVI complete")


[Step 13] Training scANVI branches...

  -> Initializing scANVI branch: reference_fine_labels
     labels_key = scanvi_labels
     unique labels = 39
scanvi_labels
Unknown                                 145335
cd16plus_nk_cells_c0                      5023
cd16plus_nk_cells_c1                      4215
trm_cytotoxic_t_cells_c0                  4058
trm_cytotoxic_t_cells_c1                  2993
cd8plus_trm_cytotoxic_t_cells_c0          2509
trm_cytotoxic_t_cells_c2                  2449
tem_effector_helper_t_cells_c0            2005
tem_trm_cytotoxic_t_cells_c0              1932
tem_effector_helper_t_cells_c1            1840
cd16plus_nk_cells_c2                      1725
tem_trm_cytotoxic_t_cells_c1              1720
nk_cells_c0                               1590
cd8plus_trm_cytotoxic_t_cells_c1          1569
cd8plus_tem_trm_cytotoxic_t_cells_c0      1337
Name: count, dtype: int64
ScanVI Model with the following params: 
unlabeled_category: Unknown, n_hidden: 128, n_latent: 100, n_la

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 166/200:  83%|████████▎ | 166/200 [2:02:47<25:09, 44.39s/it, v_num=1, train_loss_step=655, train_loss_epoch=650]  
Monitored metric elbo_validation did not improve in the last 20 records. Best score: 671.132. Signaling Trainer to stop.
  -> scANVI complete: reference_fine_labels

  -> Initializing scANVI branch: celltypist_cd4cd8
     labels_key = scanvi_labels_celltypist_cd4cd8
     unique labels = 42
scanvi_labels_celltypist_cd4cd8
Trm cytotoxic T cells               21826
Tem/Trm cytotoxic T cells           20882
CD4+ Trm cytotoxic T cells          20185
CD4+ Regulatory T cells             18548
CD16+ NK cells                      17216
CD4+ Tem/Trm cytotoxic T cells      16470
CD4+ Tem/Effector helper T cells    15748
CD16- NK cells                       8730
Tem/Temra cytotoxic T cells          8140
Regulatory T cells                   6350
NK cells                             5322
CD4+ Tcm/Naive helper T cells        5078
CD4+ Type 17 helper T cells          4634
CD4+ Tem/T

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [2:40:34<00:00, 47.49s/it, v_num=1, train_loss_step=606, train_loss_epoch=658]  

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [2:40:34<00:00, 48.17s/it, v_num=1, train_loss_step=606, train_loss_epoch=658]
  -> scANVI complete: celltypist_cd4cd8
  -> Dual scANVI complete


## Cell 17 — Step 14: Export Results

In [40]:
print("\n[Step 14] Exporting Results...")

scanvi_branch_results = {}
scanvi_branch_results["reference"] = export_scanvi_branch_results(
    scanvi_model,
    adata_train,
    adata_merged,
    result_suffix="reference",
    labels_key="scanvi_labels",
    set_as_default=True,
    compute_novelty=True,
 )

scanvi_branch_results[CELLTYPIST_SCANVI_RESULT_KEY] = export_scanvi_branch_results(
    scanvi_model_celltypist_cd4cd8,
    adata_train,
    adata_merged,
    result_suffix=CELLTYPIST_SCANVI_RESULT_KEY,
    labels_key=CELLTYPIST_SCANVI_LABEL_KEY,
    set_as_default=False,
    compute_novelty=False,
 )

print("  -> Reference-label scANVI predictions:")
print(adata_merged.obs["scanvi_pred_reference"].value_counts().head(15))

print(f"\n  -> {CELLTYPIST_SCANVI_RESULT_KEY} scANVI predictions:")
print(adata_merged.obs[f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}"].value_counts().head(15))


[Step 14] Exporting Results...


  -> Computing novelty scores (entropy-based)...
    High novelty query cells: 9856
  -> Reference-label scANVI predictions:
scanvi_pred_reference
trm_cytotoxic_t_cells_c2            70774
cd16plus_nk_cells_c0                27841
cd8plus_trm_cytotoxic_t_cells_c0    19792
regulatory_t_cells_c0               12276
cd8plus_trm_cytotoxic_t_cells_c1     8807
trm_cytotoxic_t_cells_c1             8093
tem_temra_cytotoxic_t_cells_c0       7387
regulatory_t_cells_c1                5539
gamma-delta_t_cells_c0               3679
cd16-_nk_cells_c0                    3602
cd16plus_nk_cells_c1                 3339
type_17_helper_t_cells_c0            2510
tem_effector_helper_t_cells_c1       2245
tem_trm_cytotoxic_t_cells_c1         1929
tem_trm_cytotoxic_t_cells_c0         1550
Name: count, dtype: int64

  -> celltypist_cd4cd8 scANVI predictions:
scanvi_pred_celltypist_cd4cd8
Trm cytotoxic T cells               32009
CD4+ Trm cytotoxic T cells          27320
CD4+ Tem/Effector helper T cells    255

## Cell 18 — Step 15: Attach .raw

In [41]:
print("\n[Step 15] Attaching .raw...")
from anndata import AnnData

adata_merged.raw = AnnData(X=full_counts, obs=adata_merged.obs.copy(), var=raw_var)
print(f"  OK .raw: {adata_merged.raw.n_vars} genes")


[Step 15] Attaching .raw...


  OK .raw: 33749 genes


## Cell 19 — Step 16: Multiple UMAPs (scVI + scANVI)

In [42]:
print("\n[Step 16] Computing Multiple UMAPs...")

print("  -> Getting scVI latent representation...")

# --- BUG-3 FIX ---
# Old (fragile):
#   adata_merged.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_train)
# get_latent_representation returns a numpy array ordered by adata_train.obs_names.
# Assigning it directly to adata_merged.obsm assumes the cell order is identical
# between adata_train and adata_merged, which happens to be true here but is not
# guaranteed (e.g., if sc.concat internally reorders cells). This is inconsistent
# with the index-aligned pattern already used for X_scANVI above.
# New (correct): wrap in DataFrame and reindex to adata_merged.obs_names.
latent_scvi = scvi_model.get_latent_representation(adata_train)
latent_scvi_df = pd.DataFrame(
    latent_scvi,
    index=adata_train.obs_names,
    columns=[f"scVI_{i}" for i in range(latent_scvi.shape[1])]
)
latent_scvi_aligned = latent_scvi_df.reindex(adata_merged.obs_names)
if latent_scvi_aligned.isna().any().any():
    raise ValueError("CRITICAL: Missing scVI latent representation after reindex!")
adata_merged.obsm["X_scVI"] = latent_scvi_aligned.values
# --- END FIX ---

# Run query-only Leiden clustering for novelty detection
run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION)

# 16a: UMAP on scVI latent
print("  -> Computing UMAP on scVI latent...")
sc.pp.neighbors(adata_merged, use_rep="X_scVI", n_neighbors=30, random_state=RANDOM_SEED,
                key_added="neighbors_scVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scVI")
adata_merged.obsm["X_umap_scVI"] = adata_merged.obsm["X_umap"].copy()
print(f"     Saved to X_umap_scVI")

# 16b: UMAP on scANVI latent (DEFAULT)
print("  -> Computing UMAP on scANVI latent (DEFAULT)...")
sc.pp.neighbors(adata_merged, use_rep="X_scANVI", n_neighbors=30, random_state=RANDOM_SEED,
                key_added="neighbors_scANVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scANVI")
adata_merged.obsm["X_umap_scANVI"] = adata_merged.obsm["X_umap"].copy()
adata_merged.obsm["X_umap"] = adata_merged.obsm["X_umap_scANVI"].copy()
print(f"     Saved to X_umap_scANVI and X_umap (default)")

umap_op_scanvi = UMAP(n_neighbors=30, n_components=2, min_dist=0.5, spread=1.0,
                      metric="euclidean", random_state=RANDOM_SEED)
umap_op_scanvi.fit(adata_merged.obsm["X_scANVI"])
joblib.dump(umap_op_scanvi, output_dir / f"{OUTPUT_PREFIX}_umap_scanvi_operator.joblib")
print(f"  -> UMAP operator (scANVI) saved")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(adata_merged.obs["scanvi_confidence"], bins=50, edgecolor="black")
ax.set_xlabel("Prediction Confidence")
ax.set_ylabel("Cell Count")
ax.set_title("scANVI Confidence Distribution")
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_confidence_histogram.png", dpi=150)
plt.close()


[Step 16] Computing Multiple UMAPs...
  -> Getting scVI latent representation...



[Novelty Detection] Running query-only Leiden (resolution=1.0)...
  -> Found 17 query-only clusters
leiden_query
N/A      44833
qry_0    20937
qry_1    19708
qry_2    15870
qry_3    15094
qry_4    12609
qry_5    11409
qry_6    10596
qry_7     9036
qry_8     7784
Name: count, dtype: int64
  -> Computing UMAP on scVI latent...
     Saved to X_umap_scVI
  -> Computing UMAP on scANVI latent (DEFAULT)...
     Saved to X_umap_scANVI and X_umap (default)
  -> UMAP operator (scANVI) saved


## Cell 20 — Step 17: Save Results

In [43]:
print("\n[Step 17] Saving Results...")

# Backward-compatible default model path = reference-label scANVI
scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model", overwrite=True)
scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model_reference", overwrite=True)
scanvi_model_celltypist_cd4cd8.save(
    output_dir / f"{OUTPUT_PREFIX}_scanvi_model_{CELLTYPIST_SCANVI_RESULT_KEY}",
    overwrite=True,
 )
scvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scvi_model", overwrite=True)

config = {
    "version": "1.3_dual_scanvi",
    "timestamp": datetime.now().isoformat(),
    "input": {"reference": REFERENCE_H5AD, "query": QUERY_H5AD},
    "n_hvg": int(n_hvg_final),
    "scanvi_labels": list(adata_merged.uns["scanvi_label_order"]),
    "scanvi_runs": {
        "reference": {
            "labels_key": "scanvi_labels",
            "prediction_key": "scanvi_pred_reference",
            "confidence_key": "scanvi_confidence_reference",
            "latent_key": "X_scANVI_reference",
            "probability_key": "scanvi_proba_reference",
            "default_aliases": ["scanvi_pred", "scanvi_confidence", "X_scANVI", "scanvi_proba"],
        },
        CELLTYPIST_SCANVI_RESULT_KEY: {
            "labels_key": CELLTYPIST_SCANVI_LABEL_KEY,
            "prediction_key": f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "confidence_key": f"scanvi_confidence_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "latent_key": f"X_scANVI_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "probability_key": f"scanvi_proba_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "source_label_key": "celltypist_scanvi_source_label",
            "combined_label_key": "celltypist_scanvi_combined_label",
        },
    },
    "tcell_specific": {
        "cd4_markers": CD4_MARKERS,
        "cd8_markers": CD8_MARKERS,
        "cd4_score_threshold": CD4_SCORE_THRESHOLD,
        "cd8_score_threshold": CD8_SCORE_THRESHOLD,
        "forced_markers_count": len(FORCED_MARKERS),
        "celltypist_scanvi_use_majority": CELLTYPIST_SCANVI_USE_MAJORITY,
        "celltypist_scanvi_keep_dp_dn_unprefixed": CELLTYPIST_SCANVI_KEEP_DP_DN_UNPREFIXED,
    },
    "umap_spaces": {
        "X_umap": "DEFAULT - reference-label scANVI based UMAP",
        "X_umap_scVI": "scVI latent space UMAP",
        "X_umap_scANVI": "reference-label scANVI latent space UMAP",
    },
    "bug_fixes_v1_3": {
        "BUG1_P0": "score_genes replaced with direct mean-expression in compute_module_score_efficient",
        "BUG2_P0": "celltypist model load uses local-path check before calling download_models",
        "BUG3_P1": "X_scVI uses pandas reindex for index-aligned writeback",
        "BUG4_P1": "query_mask.values used when indexing numpy obsm array",
        "BUG5_P1": "predict(soft=True) wrapped in np.asarray() for version-safe ndarray",
        "BUG6_P0": "sparse[:1000] validation uses toarray().ravel() before integer-count checks",
    }
}

with open(output_dir / f"{OUTPUT_PREFIX}_config.json", "w") as f:
    json.dump(config, f, indent=2)

output_h5ad = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"
adata_merged.write_h5ad(output_h5ad, compression="gzip")
print(f"  -> Saved: {output_h5ad}")

train_h5ad = output_dir / f"{OUTPUT_PREFIX}_train_HVG.h5ad"
adata_train.write_h5ad(train_h5ad, compression="gzip")
print(f"  -> Saved: {train_h5ad}")


[Step 17] Saving Results...


... storing 'cell_type_coarse' as categorical
... storing 'cell_type_fine' as categorical
... storing 'cd4_cd8_by_score' as categorical
... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'celltypist_scanvi_source_label' as categorical
... storing 'celltypist_scanvi_combined_label' as categorical
... storing 'scanvi_pred_reference' as categorical
... storing 'scanvi_pred' as categorical
... storing 'scanvi_pred_celltypist_cd4cd8' as categorical
... storing 'leiden_query' as categorical
... storing 'symbol_base' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260225/tcell_only_merged_pipeline/tcell_only_merged_results.h5ad


... storing 'cell_type_coarse' as categorical
... storing 'cell_type_fine' as categorical
... storing 'cd4_cd8_by_score' as categorical
... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'celltypist_scanvi_source_label' as categorical
... storing 'celltypist_scanvi_combined_label' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260225/tcell_only_merged_pipeline/tcell_only_merged_train_HVG.h5ad


In [46]:
print("\n[Step 17b] Writing logs and AnnData structure summary...")


def summarize_anndata_structure(adata, name, max_obs_cols=30, max_var_cols=20, max_uns_keys=40):
    lines = []
    lines.append("=" * 100)
    lines.append(f"AnnData summary: {name}")
    lines.append("=" * 100)
    lines.append(f"shape: {adata.n_obs:,} obs x {adata.n_vars:,} vars")
    lines.append(
        f"X: type={type(adata.X).__name__}, shape={getattr(adata.X, 'shape', 'NA')}, "
        f"dtype={getattr(adata.X, 'dtype', 'NA')}, sparse={issparse(adata.X)}"
    )

    lines.append(f"obs columns ({adata.obs.shape[1]}): {list(adata.obs.columns)}")
    for col in adata.obs.columns[:max_obs_cols]:
        series = adata.obs[col]
        try:
            n_unique = series.nunique(dropna=False)
        except Exception:
            n_unique = "NA"
        lines.append(f"  - obs[{col!r}]: dtype={series.dtype}, n_unique={n_unique}")
    if adata.obs.shape[1] > max_obs_cols:
        lines.append(f"  ... ({adata.obs.shape[1] - max_obs_cols} more obs columns)")

    lines.append(f"var columns ({adata.var.shape[1]}): {list(adata.var.columns)}")
    for col in adata.var.columns[:max_var_cols]:
        series = adata.var[col]
        try:
            n_unique = series.nunique(dropna=False)
        except Exception:
            n_unique = "NA"
        lines.append(f"  - var[{col!r}]: dtype={series.dtype}, n_unique={n_unique}")
    if adata.var.shape[1] > max_var_cols:
        lines.append(f"  ... ({adata.var.shape[1] - max_var_cols} more var columns)")

    layer_keys = list(adata.layers.keys())
    lines.append(f"layers ({len(layer_keys)}): {layer_keys}")
    for key in layer_keys:
        layer = adata.layers[key]
        lines.append(
            f"  - layers[{key!r}]: type={type(layer).__name__}, "
            f"shape={getattr(layer, 'shape', 'NA')}, dtype={getattr(layer, 'dtype', 'NA')}"
        )

    obsm_keys = list(adata.obsm.keys())
    lines.append(f"obsm ({len(obsm_keys)}): {obsm_keys}")
    for key in obsm_keys:
        value = adata.obsm[key]
        lines.append(
            f"  - obsm[{key!r}]: type={type(value).__name__}, shape={getattr(value, 'shape', 'NA')}"
        )

    varm_keys = list(adata.varm.keys())
    lines.append(f"varm ({len(varm_keys)}): {varm_keys}")
    for key in varm_keys:
        value = adata.varm[key]
        lines.append(
            f"  - varm[{key!r}]: type={type(value).__name__}, shape={getattr(value, 'shape', 'NA')}"
        )

    obsp_keys = list(adata.obsp.keys())
    lines.append(f"obsp ({len(obsp_keys)}): {obsp_keys}")
    for key in obsp_keys:
        value = adata.obsp[key]
        lines.append(
            f"  - obsp[{key!r}]: type={type(value).__name__}, shape={getattr(value, 'shape', 'NA')}"
        )

    uns_keys = list(adata.uns.keys())
    lines.append(f"uns keys ({len(uns_keys)}): {uns_keys[:max_uns_keys]}")
    if len(uns_keys) > max_uns_keys:
        lines.append(f"  ... ({len(uns_keys) - max_uns_keys} more uns keys)")

    if adata.raw is not None:
        lines.append(
            f"raw: present, shape={adata.raw.shape}, var_columns={list(adata.raw.var.columns)}"
        )
    else:
        lines.append("raw: None")

    return "\n".join(lines)


def summarize_h5ad_on_disk(h5ad_path, name):
    h5ad_path = Path(h5ad_path)
    lines = []
    lines.append("-" * 100)
    lines.append(f"Saved h5ad summary: {name}")
    lines.append(f"path: {h5ad_path}")
    if not h5ad_path.exists():
        lines.append("status: missing")
        return "\n".join(lines)

    lines.append(f"size_bytes: {h5ad_path.stat().st_size:,}")
    backed = sc.read_h5ad(h5ad_path, backed="r")
    try:
        lines.append(f"shape: {backed.n_obs:,} obs x {backed.n_vars:,} vars")
        lines.append(f"obs columns ({backed.obs.shape[1]}): {list(backed.obs.columns)}")
        lines.append(f"var columns ({backed.var.shape[1]}): {list(backed.var.columns)}")
        lines.append(f"layers: {list(backed.layers.keys())}")
        lines.append(f"obsm: {list(backed.obsm.keys())}")
        lines.append(f"varm: {list(backed.varm.keys())}")
        lines.append(f"obsp: {list(backed.obsp.keys())}")
        lines.append(f"uns keys: {list(backed.uns.keys())[:40]}")
        lines.append(f"raw present: {backed.raw is not None}")
    finally:
        if getattr(backed, 'file', None) is not None:
            backed.file.close()
    return "\n".join(lines)


run_timestamp = datetime.now().isoformat()
run_log_path = output_dir / f"{OUTPUT_PREFIX}_run_log.txt"
structure_log_path = output_dir / f"{OUTPUT_PREFIX}_anndata_structure.txt"

run_log_lines = [
    "=" * 100,
    f"Run timestamp: {run_timestamp}",
    f"Reference H5AD: {REFERENCE_H5AD}",
    f"Query H5AD: {QUERY_H5AD}",
    f"Merged output H5AD: {output_h5ad}",
    f"Train output H5AD: {train_h5ad}",
    f"adata_merged shape: {adata_merged.shape}",
    f"adata_train shape: {adata_train.shape}",
    f"scVI latent keys: {[k for k in adata_merged.obsm.keys() if 'scVI' in k or 'scANVI' in k]}",
    f"obs prediction keys: {[k for k in adata_merged.obs.columns if 'scanvi' in k.lower() or 'celltypist' in k.lower()]}",
]

with open(run_log_path, "a", encoding="utf-8") as f:
    f.write("\n".join(run_log_lines) + "\n")

structure_sections = [
    summarize_anndata_structure(adata_merged, "adata_merged (in memory)"),
    summarize_anndata_structure(adata_train, "adata_train (in memory)"),
    summarize_h5ad_on_disk(output_h5ad, "merged results h5ad"),
    summarize_h5ad_on_disk(train_h5ad, "train HVG h5ad"),
]

with open(structure_log_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(structure_sections) + "\n")

print(f"  -> Appended run log: {run_log_path}")
print(f"  -> Wrote AnnData structure summary: {structure_log_path}")
print("  -> Done")


[Step 17b] Writing logs and AnnData structure summary...
  -> Appended run log: /home/h2048/data/py/20260225/tcell_only_merged_pipeline/tcell_only_merged_run_log.txt
  -> Wrote AnnData structure summary: /home/h2048/data/py/20260225/tcell_only_merged_pipeline/tcell_only_merged_anndata_structure.txt
  -> Done


## Cell 21 — Step 18: T Cell Specific Visualization

In [44]:
print("\n[Step 18] Creating T cell visualizations...")

fig = plt.figure(figsize=(24, 20))
gs = fig.add_gridspec(5, 4, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
sc.pl.umap(adata_merged, color="data_source", ax=ax1, show=False, title="Data Source (scANVI)", s=15)

ax2 = fig.add_subplot(gs[0, 1])
adata_merged.obs["_ref_coarse"] = pd.Series(pd.NA, index=adata_merged.obs_names, dtype="object")
mask_ref = adata_merged.obs["data_source"] == "reference"
adata_merged.obs.loc[mask_ref, "_ref_coarse"] = adata_merged.obs.loc[mask_ref, "cell_type_coarse"].astype(str).values
adata_merged.obs["_ref_coarse"] = adata_merged.obs["_ref_coarse"].astype("category")
sc.pl.umap(adata_merged, color="_ref_coarse", ax=ax2, show=False, title="Reference Coarse Labels", legend_loc="on data", s=15)

ax3 = fig.add_subplot(gs[0, 2])
sc.pl.umap(adata_merged, color="scanvi_pred", ax=ax3, show=False, title="scANVI Predictions", legend_loc="on data", s=15)

ax4 = fig.add_subplot(gs[0, 3])
sc.pl.umap(adata_merged, color="scanvi_confidence", ax=ax4, show=False, title="Confidence", cmap="viridis", vmin=0, vmax=1, s=15)

ax5 = fig.add_subplot(gs[1, 0])
sc.pl.umap(adata_merged, color="CD4_score", ax=ax5, show=False, title="CD4 Module Score", cmap="Reds", s=15)

ax6 = fig.add_subplot(gs[1, 1])
sc.pl.umap(adata_merged, color="CD8_score", ax=ax6, show=False, title="CD8 Module Score", cmap="Blues", s=15)

ax7 = fig.add_subplot(gs[1, 2])
sc.pl.umap(adata_merged, color="cd4_cd8_by_score", ax=ax7, show=False, title="CD4/CD8 by Score", legend_loc="on data", s=15)

ax8 = fig.add_subplot(gs[1, 3])
query_mask_viz = adata_merged.obs["data_source"] == "query"
ax8.scatter(
    adata_merged.obs.loc[query_mask_viz, "CD4_score"],
    adata_merged.obs.loc[query_mask_viz, "CD8_score"],
    c=adata_merged.obs.loc[query_mask_viz, "scanvi_confidence"],
    cmap="viridis", s=5, alpha=0.5
)
ax8.set_xlabel("CD4 Score")
ax8.set_ylabel("CD8 Score")
ax8.set_title(f"Query: CD4 vs CD8 Score\n(threshold={CD4_SCORE_THRESHOLD})")
ax8.axhline(y=CD8_SCORE_THRESHOLD, color='k', linestyle='--', alpha=0.3)
ax8.axvline(x=CD4_SCORE_THRESHOLD, color='k', linestyle='--', alpha=0.3)

ax9 = fig.add_subplot(gs[2, 0])
if "CD3E" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="CD3E", ax=ax9, show=False, title="CD3E (pan-T)", cmap="Reds", s=15, use_raw=True)

ax10 = fig.add_subplot(gs[2, 1])
if "CD4" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="CD4", ax=ax10, show=False, title="CD4", cmap="Reds", s=15, use_raw=True)

ax11 = fig.add_subplot(gs[2, 2])
if "CD8A" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="CD8A", ax=ax11, show=False, title="CD8A", cmap="Reds", s=15, use_raw=True)

ax12 = fig.add_subplot(gs[2, 3])
if "CD8B" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="CD8B", ax=ax12, show=False, title="CD8B", cmap="Reds", s=15, use_raw=True)

ax13 = fig.add_subplot(gs[3, 0])
if "CCR7" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="CCR7", ax=ax13, show=False, title="CCR7 (Naive/CM)", cmap="Reds", s=15, use_raw=True)

ax14 = fig.add_subplot(gs[3, 1])
if "FOXP3" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="FOXP3", ax=ax14, show=False, title="FOXP3 (Treg)", cmap="Reds", s=15, use_raw=True)

ax15 = fig.add_subplot(gs[3, 2])
if "GZMB" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="GZMB", ax=ax15, show=False, title="GZMB (Effector)", cmap="Reds", s=15, use_raw=True)

ax16 = fig.add_subplot(gs[3, 3])
if "MKI67" in adata_merged.raw.var_names:
    sc.pl.umap(adata_merged, color="MKI67", ax=ax16, show=False, title="MKI67 (Proliferating)", cmap="Reds", s=15, use_raw=True)

ax17 = fig.add_subplot(gs[4, 0])
ref_cd4cd8 = adata_merged.obs.loc[adata_merged.obs["data_source"] == "reference", "cell_type_coarse"].value_counts()
qry_cd4cd8 = adata_merged.obs.loc[adata_merged.obs["data_source"] == "query", "cd4_cd8_by_score"].value_counts()
x = np.arange(len(ref_cd4cd8.index))
width = 0.35
ax17.bar(x - width/2, ref_cd4cd8.values, width, label="Reference (annotated)", alpha=0.8)
ax17.bar(x + width/2, [qry_cd4cd8.get(k, 0) for k in ref_cd4cd8.index], width, label="Query (by score)", alpha=0.8)
ax17.set_xticks(x)
ax17.set_xticklabels(ref_cd4cd8.index, rotation=45, ha="right")
ax17.set_ylabel("Cell Count")
ax17.set_title("CD4/CD8 Distribution")
ax17.legend()

ax18 = fig.add_subplot(gs[4, 1])
sc.pl.umap(adata_merged, color="novelty_score", ax=ax18, show=False,
           title="Novelty Score (Query)", cmap="hot", vmin=0, vmax=1, s=15)

ax19 = fig.add_subplot(gs[4, 2])
conf_ref = adata_merged.obs.loc[adata_merged.obs["data_source"] == "reference", "scanvi_confidence"]
conf_qry = adata_merged.obs.loc[adata_merged.obs["data_source"] == "query", "scanvi_confidence"]
ax19.hist([conf_ref, conf_qry], bins=30, label=["Reference", "Query"], alpha=0.7)
ax19.set_xlabel("Confidence")
ax19.set_ylabel("Cell Count")
ax19.set_title("Confidence Distribution by Source")
ax19.legend()

ax20 = fig.add_subplot(gs[4, 3])
sc.pl.umap(adata_merged, color="leiden_query", ax=ax20, show=False,
           title="Query-only Leiden Clusters", legend_loc="on data", s=15)

plt.savefig(output_dir / f"{OUTPUT_PREFIX}_tcell_overview.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_tcell_overview.pdf")

print("  -> Creating scVI vs scANVI UMAP comparison...")
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))

sc.pl.umap(adata_merged, basis="X_umap_scVI", color="data_source", ax=axes2[0, 0], show=False,
           title="Data Source (scVI UMAP)", s=10)
sc.pl.umap(adata_merged, basis="X_umap_scVI", color="_ref_coarse", ax=axes2[0, 1], show=False,
           title="Reference Labels (scVI)", legend_loc="on data", s=10)
sc.pl.umap(adata_merged, basis="X_umap_scVI", color="scanvi_pred", ax=axes2[0, 2], show=False,
           title="scANVI Predictions (scVI)", legend_loc="on data", s=10)

sc.pl.umap(adata_merged, basis="X_umap_scANVI", color="data_source", ax=axes2[1, 0], show=False,
           title="Data Source (scANVI UMAP)", s=10)
sc.pl.umap(adata_merged, basis="X_umap_scANVI", color="_ref_coarse", ax=axes2[1, 1], show=False,
           title="Reference Labels (scANVI)", legend_loc="on data", s=10)
sc.pl.umap(adata_merged, basis="X_umap_scANVI", color="scanvi_pred", ax=axes2[1, 2], show=False,
           title="scANVI Predictions (scANVI)", legend_loc="on data", s=10)

plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_umap_comparison.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_umap_comparison.pdf")

print("  -> Creating novelty analysis report...")
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 12))

qry_mask = adata_merged.obs["data_source"] == "query"

ax = axes3[0, 0]
ax.hist(adata_merged.obs.loc[qry_mask, "novelty_score"], bins=50, edgecolor="black", alpha=0.7)
ax.axvline(x=0.7, color='r', linestyle='--', label='High novelty threshold')
ax.set_xlabel("Novelty Score")
ax.set_ylabel("Cell Count")
ax.set_title("Query Cells: Novelty Score Distribution")
ax.legend()

ax = axes3[0, 1]
scatter = ax.scatter(
    adata_merged.obs.loc[qry_mask, "scanvi_confidence"],
    adata_merged.obs.loc[qry_mask, "novelty_score"],
    c=adata_merged.obs.loc[qry_mask, "scanvi_entropy"],
    cmap="viridis", s=5, alpha=0.5
)
ax.set_xlabel("scANVI Confidence")
ax.set_ylabel("Novelty Score")
ax.set_title("Query: Confidence vs Novelty (colored by entropy)")
plt.colorbar(scatter, ax=ax)

ax = axes3[1, 0]
leiden_counts = adata_merged.obs.loc[qry_mask, "leiden_query"].value_counts().head(15)
ax.barh(range(len(leiden_counts)), leiden_counts.values)
ax.set_yticks(range(len(leiden_counts)))
ax.set_yticklabels(leiden_counts.index)
ax.set_xlabel("Cell Count")
ax.set_title("Query-only Leiden Clusters (Top 15)")

ax = axes3[1, 1]
qry_data = adata_merged.obs.loc[qry_mask]
mismatch = []
for idx, row in qry_data.iterrows():
    score_type = row["cd4_cd8_by_score"]
    pred = row["scanvi_pred"]
    if "CD4" in pred and score_type == "CD8_single":
        mismatch.append("PredCD4/ScoreCD8")
    elif "CD8" in pred and score_type == "CD4_single":
        mismatch.append("PredCD8/ScoreCD4")
    elif "CD4" in pred and score_type == "CD4_single":
        mismatch.append("Match_CD4")
    elif "CD8" in pred and score_type == "CD8_single":
        mismatch.append("Match_CD8")
    else:
        mismatch.append("Other/Unclear")
mismatch_series = pd.Series(mismatch)
mismatch_counts = mismatch_series.value_counts()
ax.pie(mismatch_counts.values, labels=mismatch_counts.index, autopct='%1.1f%%')
ax.set_title("CD4/CD8: Prediction vs Score Agreement")

plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_novelty_analysis.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_novelty_analysis.pdf")

adata_merged.obs.drop(columns=["_ref_coarse"], inplace=True, errors="ignore")

# --- Summary ---
print("\n" + "=" * 80)
print("T CELL PIPELINE COMPLETE (v1.2)")
print("=" * 80)
print(f"\nOutput: {output_dir}")
print(f"\nKey observations:")
print(f"  - Total cells: {adata_merged.n_obs:,}")
print(f"  - Reference: {(adata_merged.obs['data_source'] == 'reference').sum():,}")
print(f"  - Query: {(adata_merged.obs['data_source'] == 'query').sum():,}")

print(f"\nCD4/CD8 Distribution in Query (by score):")
print(adata_merged.obs.loc[adata_merged.obs["data_source"] == "query", "cd4_cd8_by_score"].value_counts())

print(f"\nTop scANVI predictions in Query:")
print(adata_merged.obs.loc[adata_merged.obs["data_source"] == "query", "scanvi_pred"].value_counts().head(10))

print(f"\nNovelty Detection Summary:")
print(f"  - High novelty cells (score > 0.7): {adata_merged.obs['is_potentially_novel'].sum():,}")
print(f"  - Query-only Leiden clusters: {adata_merged.obs.loc[qry_mask, 'leiden_query'].nunique()}")

print(f"\nUMAP spaces available:")
print(f"  - X_umap: DEFAULT (scANVI-based)")
print(f"  - X_umap_scVI: scVI latent space")
print(f"  - X_umap_scANVI: scANVI latent space")

print("\n" + "=" * 80)
print("Bug Fixes Applied in v1.2:")
print("=" * 80)
print("  BUG1 [P0] score_genes no-background -> direct mean-expression score")
print("  BUG2 [P0] download_models full-path -> local-file-check + basename")
print("  BUG3 [P1] X_scVI direct assignment -> pandas reindex pattern")
print("  BUG4 [P1] query_mask Series indexing -> query_mask.values")
print("  BUG5 [P1] predict(soft=True) -> np.asarray() for version safety")
print("=" * 80)


[Step 18] Creating T cell visualizations...
  -> Saved: tcell_only_merged_tcell_overview.pdf
  -> Creating scVI vs scANVI UMAP comparison...


TypeError: embedding() got multiple values for argument 'basis'

## Cell 22 — (Optional) Run as script
If you want to run the whole pipeline as a function, uncomment below.

In [ ]:
# main()  # uncomment to run everything at once